In [4]:
# Ячейка 0: Диагностика GPU и очистка памяти
import torch
import os
import subprocess
import gc

print("="*50)
print("ДИАГНОСТИКА GPU")
print("="*50)

# Проверка доступности CUDA
print(f"CUDA доступен: {torch.cuda.is_available()}")
print(f"Версия CUDA: {torch.version.cuda}")
print(f"Количество GPU: {torch.cuda.device_count()}")

# Информация о каждом GPU
for i in range(torch.cuda.device_count()):
    print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  Всего памяти: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")
    print(f"  Текущая выделенная память: {torch.cuda.memory_allocated(i) / 1e6:.2f} MB")
    print(f"  Зарезервированная память: {torch.cuda.memory_reserved(i) / 1e6:.2f} MB")

# Системная информация
print("\n" + "="*50)
print("СИСТЕМНАЯ ИНФОРМАЦИЯ")
print("="*50)

# Информация о видеокартах через nvidia-smi
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used,memory.free', '--format=csv'], 
                          capture_output=True, text=True)
    print("nvidia-smi вывод:")
    print(result.stdout)
except:
    print("nvidia-smi недоступен")

# Информация о процессах
print("\n" + "="*50)
print("ИНФОРМАЦИЯ О ПРОЦЕССАХ")
print("="*50)

# Очистка памяти
print("\nОчистка памяти...")
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"Память после очистки:")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.memory_allocated(i) / 1e6:.2f} MB выделено, "
          f"{torch.cuda.memory_reserved(i) / 1e6:.2f} MB зарезервировано")

# Проверка, какое устройство выберет PyTorch по умолчанию
print(f"\nУстройство по умолчанию: {torch.cuda.current_device() if torch.cuda.is_available() else 'CPU'}")
print(f"Текущее устройство: {torch.cuda.current_device()}")

print("\n" + "="*50)
print("ГОТОВНОСТЬ К ЗАПУСКУ")
print("="*50)

if torch.cuda.is_available():
    print("✅ GPU доступен! Можно запускать обучение.")
    print(f"Рекомендуемые настройки:")
    print(f"  - BATCH_SIZE = 16 или 24 (вместо 32)")
    print(f"  - NUM_WORKERS = 2")
    print(f"  - Убедитесь, что DEVICE = torch.device('cuda')")
else:
    print("❌ GPU НЕ доступен! Проверьте настройки Kaggle:")
    print("1. Нажмите на 'Settings' в правой панели")
    print("2. В разделе 'Accelerator' выберите 'GPU T4 x2'")
    print("3. Нажмите 'Save' и перезапустите ноутбук")

# Тестовый тензор на GPU
if torch.cuda.is_available():
    try:
        test_tensor = torch.randn(100, 100).cuda()
        print(f"\n✅ Тестовый тензор успешно создан на GPU: {test_tensor.device}")
        del test_tensor
    except Exception as e:
        print(f"\n❌ Ошибка при создании тензора на GPU: {e}")

ДИАГНОСТИКА GPU
CUDA доступен: True
Версия CUDA: 12.6
Количество GPU: 1

GPU 0: Tesla P100-PCIE-16GB
  Всего памяти: 17.06 GB
  Текущая выделенная память: 16692.89 MB
  Зарезервированная память: 16703.82 MB

СИСТЕМНАЯ ИНФОРМАЦИЯ
nvidia-smi вывод:
name, memory.total [MiB], memory.used [MiB], memory.free [MiB]
Tesla P100-PCIE-16GB, 16384 MiB, 16243 MiB, 28 MiB


ИНФОРМАЦИЯ О ПРОЦЕССАХ

Очистка памяти...
Память после очистки:
GPU 0: 15283.26 MB выделено, 15441.33 MB зарезервировано

Устройство по умолчанию: 0
Текущее устройство: 0

ГОТОВНОСТЬ К ЗАПУСКУ
✅ GPU доступен! Можно запускать обучение.
Рекомендуемые настройки:
  - BATCH_SIZE = 16 или 24 (вместо 32)
  - NUM_WORKERS = 2
  - Убедитесь, что DEVICE = torch.device('cuda')

✅ Тестовый тензор успешно создан на GPU: cuda:0


In [ ]:
    import os
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import Dataset, DataLoader
    import torchvision.transforms as transforms
    import pandas as pd
    import numpy as np
    from PIL import Image
    from sklearn.model_selection import StratifiedKFold
    from tqdm import tqdm
    import random
    from pathlib import Path
    import timm
    from timm.loss import SoftTargetCrossEntropy
    import warnings
    import gc
    warnings.filterwarnings('ignore')
    
    # Настройка окружения для лучшего использования памяти
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    
    # Константы с оптимизированными параметрами
    CLASS_NAMES = [
        "Апельсин", "Бананы", "Груши", "Кабачки", "Капуста", 
        "Картофель", "Киви", "Лимон", "Лук", "Мандарины", 
        "Морковь", "Огурцы", "Томаты", "Яблоки зелёные", "Яблоки красные"
    ]
    NUM_CLASSES = len(CLASS_NAMES)
    IMG_SIZE = 256  # Еще уменьшим для экономии памяти
    BATCH_SIZE = 8  # Уменьшим еще больше для надежности
    EPOCHS = 15
    N_FOLDS = 5
    
    # Настройка устройства - будем использовать GPU 1, так как он свободен
    if torch.cuda.is_available():
        # Проверяем доступную память на каждом GPU
        print("Доступные GPU:")
        for i in range(torch.cuda.device_count()):
            total_memory = torch.cuda.get_device_properties(i).total_memory / 1e9
            allocated = torch.cuda.memory_allocated(i) / 1e9
            free_memory = total_memory - allocated
            print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
            print(f"    Всего: {total_memory:.2f} GB")
            print(f"    Занято: {allocated:.2f} GB")
            print(f"    Свободно: {free_memory:.2f} GB")
        
        # Выбираем GPU с наибольшей свободной памятью
        best_gpu = 0
        max_free = 0
        for i in range(torch.cuda.device_count()):
            total = torch.cuda.get_device_properties(i).total_memory
            allocated = torch.cuda.memory_allocated(i)
            free = total - allocated
            if free > max_free:
                max_free = free
                best_gpu = i
        
        DEVICE = torch.device(f'cuda:{best_gpu}')
        torch.cuda.set_device(best_gpu)
        torch.cuda.empty_cache()
        gc.collect()
        
        print(f"\nИспользуется GPU {best_gpu}: {torch.cuda.get_device_name(best_gpu)}")
        print(f"Свободная память: {max_free / 1e9:.2f} GB")
    else:
        DEVICE = torch.device('cpu')
        print("CUDA не доступна, используется CPU")
    
    # Создаем маппинг классов
    class_to_idx = {cls_name: idx for idx, cls_name in enumerate(CLASS_NAMES)}
    
    class FocalLoss(nn.Module):
        """Focal Loss с оптимизацией памяти"""
        def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
            super().__init__()
            self.alpha = alpha
            self.gamma = gamma
            self.reduction = reduction
            
        def forward(self, inputs, targets):
            # Более эффективная реализация
            log_probs = nn.functional.log_softmax(inputs, dim=1)
            probs = torch.exp(log_probs)
            
            ce_loss = -log_probs.gather(dim=1, index=targets.unsqueeze(1)).squeeze(1)
            pt = probs.gather(dim=1, index=targets.unsqueeze(1)).squeeze(1)
            
            focal_loss = ((1 - pt) ** self.gamma) * ce_loss
            
            if self.alpha is not None:
                alpha_t = self.alpha[targets]
                focal_loss = alpha_t * focal_loss
            
            if self.reduction == 'mean':
                return focal_loss.mean()
            elif self.reduction == 'sum':
                return focal_loss.sum()
            else:
                return focal_loss
    
    class CustomDataset(Dataset):
        """Кастомный датасет с оптимизацией загрузки"""
        def __init__(self, image_paths, labels, transform=None):
            self.image_paths = image_paths
            self.labels = labels
            self.transform = transform
        
        def __len__(self):
            return len(self.image_paths)
        
        def __getitem__(self, idx):
            image_path = self.image_paths[idx]
            try:
                image = Image.open(image_path).convert('RGB')
            except Exception as e:
                print(f"Ошибка загрузки {image_path}: {e}")
                image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color='black')
            
            label = self.labels[idx]
            
            if self.transform:
                image = self.transform(image)
            
            return image, label
    
    def get_train_transforms(is_train=True):
        """Трансформации с оптимизированным размером"""
        if is_train:
            return transforms.Compose([
                transforms.Resize((IMG_SIZE, IMG_SIZE)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(10),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
        else:
            return transforms.Compose([
                transforms.Resize((IMG_SIZE, IMG_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
    
    def collect_train_images(train_path):
        """Собирает все пути к изображениям"""
        image_paths = []
        labels = []
        
        train_path = Path(train_path)
        
        for class_name in CLASS_NAMES:
            class_folder = train_path / class_name
            if not class_folder.exists():
                print(f"Предупреждение: папка {class_folder} не найдена")
                continue
            
            for subfolder in class_folder.iterdir():
                if subfolder.is_dir():
                    for img_file in subfolder.glob('*.*'):
                        if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                            image_paths.append(str(img_file))
                            labels.append(class_to_idx[class_name])
        
        return image_paths, labels
    
    def calculate_class_weights(labels):
        """Вычисление весов для классов"""
        class_counts = np.bincount(labels)
        total = len(labels)
        weights = total / (len(class_counts) * class_counts)
        weights = weights / weights.sum() * len(class_counts)
        return torch.FloatTensor(weights).to(DEVICE)
    
    def get_memory_usage():
        """Получение информации об использовании памяти GPU"""
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated(DEVICE) / 1e9
            reserved = torch.cuda.memory_reserved(DEVICE) / 1e9
            return f"Alloc: {allocated:.2f}GB, Reserved: {reserved:.2f}GB"
        return "CPU mode"
    
    def train_model(train_loader, val_loader, model, criterion, optimizer, scheduler, num_epochs, fold):
        """Обучение модели с оптимизацией памяти"""
        best_val_acc = 0.0
        best_model_state = None
        
        # Используем градиентное накопление
        accumulation_steps = 4  # Эффективный batch_size = BATCH_SIZE * accumulation_steps
        
        for epoch in range(num_epochs):
            # Training phase
            model.train()
            train_loss = 0.0
            train_correct = 0
            train_total = 0
            optimizer.zero_grad()
            
            train_bar = tqdm(train_loader, desc=f'Fold {fold+1} Epoch {epoch+1}/{num_epochs} [Train]')
            for batch_idx, (images, labels) in enumerate(train_bar):
                images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss = loss / accumulation_steps
                loss.backward()
                
                if (batch_idx + 1) % accumulation_steps == 0:
                    optimizer.step()
                    optimizer.zero_grad()
                
                train_loss += loss.item() * accumulation_steps
                _, predicted = torch.max(outputs.data, 1)
                train_total += labels.size(0)
                train_correct += (predicted == labels).sum().item()
                
                # Очистка неиспользуемых переменных
                del images, labels, outputs, loss
                if batch_idx % 10 == 0:
                    torch.cuda.empty_cache()
                    gc.collect()
                
                train_bar.set_postfix({
                    'loss': f'{train_loss/train_total:.4f}',
                    'acc': f'{100.*train_correct/train_total:.2f}%',
                    'mem': get_memory_usage()
                })
            
            # Обеспечиваем применение градиентов для последнего батча
            if (batch_idx + 1) % accumulation_steps != 0:
                optimizer.step()
                optimizer.zero_grad()
            
            train_acc = 100. * train_correct / train_total
            
            # Validation phase
            model.eval()
            val_loss = 0.0
            val_correct = 0
            val_total = 0
            
            with torch.no_grad():
                val_bar = tqdm(val_loader, desc=f'Fold {fold+1} Epoch {epoch+1}/{num_epochs} [Val]')
                for images, labels in val_bar:
                    images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
                    
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    
                    val_loss += loss.item()
                    _, predicted = torch.max(outputs.data, 1)
                    val_total += labels.size(0)
                    val_correct += (predicted == labels).sum().item()
                    
                    del images, labels, outputs, loss
                    
                    val_bar.set_postfix({
                        'loss': f'{val_loss/len(val_loader):.4f}',
                        'acc': f'{100.*val_correct/val_total:.2f}%'
                    })
            
            val_acc = 100. * val_correct / val_total
            
            if scheduler:
                scheduler.step()
            
            print(f'Fold {fold+1} Epoch {epoch+1}: Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%')
            
            # Сохраняем лучшую модель
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                # Сохраняем на CPU для экономии памяти GPU
                best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            
            # Очистка памяти после каждой эпохи
            torch.cuda.empty_cache()
            gc.collect()
        
        return best_model_state, best_val_acc
    
    def predict_test_images(models, test_folder, output_csv='predictions.csv'):
        """Предсказание с использованием ансамбля"""
        test_images = []
        test_path = Path(test_folder)
        for img_file in test_path.glob('*.*'):
            if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                test_images.append(img_file)
        
        print(f"Найдено {len(test_images)} тестовых изображений")
        test_transform = get_train_transforms(is_train=False)
        
        predictions = []
        
        for img_path in tqdm(test_images, desc="Предсказание"):
            try:
                image = Image.open(img_path).convert('RGB')
                image_tensor = test_transform(image).unsqueeze(0)
                
                ensemble_probs = []
                with torch.no_grad():
                    for i, model in enumerate(models):
                        # Поочередно перемещаем модели на GPU
                        model = model.to(DEVICE)
                        image_tensor_gpu = image_tensor.to(DEVICE)
                        outputs = model(image_tensor_gpu)
                        probs = torch.softmax(outputs, dim=1).cpu()
                        ensemble_probs.append(probs)
                        
                        # Сразу перемещаем модель обратно на CPU
                        model.to('cpu')
                        torch.cuda.empty_cache()
                
                avg_probs = torch.mean(torch.stack(ensemble_probs), dim=0)
                predicted_class = torch.argmax(avg_probs, dim=1).item()
                
                predictions.append({
                    'image_id': img_path.name,
                    'label': predicted_class
                })
                
                # Очистка памяти
                del image_tensor, ensemble_probs, avg_probs
                torch.cuda.empty_cache()
                gc.collect()
                
            except Exception as e:
                print(f"Ошибка при обработке {img_path}: {e}")
                predictions.append({'image_id': img_path.name, 'label': 0})
        
        df = pd.DataFrame(predictions)
        df.to_csv(output_csv, index=False)
        print(f"Предсказания сохранены в {output_csv}")
        
        return df
    
    def main():
        print(f"Используется устройство: {DEVICE}")
        if DEVICE.type == 'cuda':
            print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")
            total_memory = torch.cuda.get_device_properties(DEVICE).total_memory / 1e9
            print(f"Всего памяти: {total_memory:.2f} GB")
            print(f"Текущее использование: {get_memory_usage()}")
        
        # Пути к данным
        train_path = "/kaggle/input/dl-lab-1-image-classification/train/train"
        test_path = "/kaggle/input/dl-lab-1-image-classification/test_images/test_images"
        
        # Проверка существования путей
        if not os.path.exists(train_path):
            print(f"ОШИБКА: Путь {train_path} не существует!")
            return
        
        # Собираем изображения
        print("Сбор обучающих изображений...")
        image_paths, labels = collect_train_images(train_path)
        
        if len(image_paths) == 0:
            print("ОШИБКА: Не найдено изображений!")
            return
        
        print(f"Всего найдено {len(image_paths)} изображений")
        
        # Вычисляем веса классов
        class_weights = calculate_class_weights(labels)
        
        # Кросс-валидация
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
        
        trained_models = []
        fold_scores = []
        
        for fold, (train_idx, val_idx) in enumerate(skf.split(image_paths, labels)):
            print(f"\n{'='*50}")
            print(f"Fold {fold+1}/{N_FOLDS}")
            print(f"{'='*50}")
            
            # Разделение данных
            train_paths = [image_paths[i] for i in train_idx]
            train_labels = [labels[i] for i in train_idx]
            val_paths = [image_paths[i] for i in val_idx]
            val_labels = [labels[i] for i in val_idx]
            
            print(f"Train samples: {len(train_paths)}, Val samples: {len(val_paths)}")
            
            # Создание датасетов
            train_dataset = CustomDataset(
                train_paths, 
                train_labels, 
                transform=get_train_transforms(is_train=True)
            )
            val_dataset = CustomDataset(
                val_paths, 
                val_labels, 
                transform=get_train_transforms(is_train=False)
            )
            
            train_loader = DataLoader(
                train_dataset, 
                batch_size=BATCH_SIZE, 
                shuffle=True, 
                num_workers=2,
                pin_memory=True,
                persistent_workers=False
            )
            val_loader = DataLoader(
                val_dataset, 
                batch_size=BATCH_SIZE, 
                shuffle=False, 
                num_workers=2,
                pin_memory=True,
                persistent_workers=False
            )
            
            # Создание модели
            print("Создание модели...")
            try:
                model = timm.create_model(
                    'convnextv2_base.fcmae_ft_in22k_in1k',
                    pretrained=True,
                    num_classes=NUM_CLASSES
                )
            except Exception as e:
                print(f"ConvNeXt V2 не доступен: {e}")
                print("Использую обычный ConvNeXt...")
                model = timm.create_model(
                    'convnext_base.fb_in22k_ft_in1k',
                    pretrained=True,
                    num_classes=NUM_CLASSES
                )
            
            model = model.to(DEVICE)
            
            # Функция потерь и оптимизатор
            criterion = FocalLoss(alpha=class_weights, gamma=2.0)
            optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
            
            # Обучение
            best_model_state, best_val_acc = train_model(
                train_loader, 
                val_loader, 
                model, 
                criterion, 
                optimizer, 
                scheduler, 
                EPOCHS, 
                fold
            )
            
            # Сохраняем модель
            model.load_state_dict(best_model_state)
            torch.save(best_model_state, f'best_model_fold_{fold+1}.pth')
            trained_models.append(model)
            fold_scores.append(best_val_acc)
            
            # Очистка памяти между фолдами
            del model, train_loader, val_loader, train_dataset, val_dataset, best_model_state
            torch.cuda.empty_cache()
            gc.collect()
            
            print(f"Fold {fold+1} лучшая точность: {best_val_acc:.2f}%")
        
        # Результаты
        print(f"\nСредняя точность: {np.mean(fold_scores):.2f}%")
        print(f"Стандартное отклонение: {np.std(fold_scores):.2f}%")
        
        # Предсказание
        if os.path.exists(test_path):
            print("\nПредсказание на тестовых изображениях...")
            predictions_df = predict_test_images(trained_models, test_path, 'test_predictions.csv')
            print("\nПервые 10 предсказаний:")
            print(predictions_df.head(10))
            
            # Показываем распределение
            print("\nРаспределение предсказаний:")
            for label in range(NUM_CLASSES):
                count = len(predictions_df[predictions_df['label'] == label])
                if count > 0:
                    print(f"  {CLASS_NAMES[label]}: {count}")
        else:
            print(f"\nПапка {test_path} не найдена. Пропускаем тестирование.")
    
    if __name__ == "__main__":
        main()

Доступные GPU:
  GPU 0: Tesla P100-PCIE-16GB
    Всего: 17.06 GB
    Занято: 16.47 GB
    Свободно: 0.59 GB

Используется GPU 0: Tesla P100-PCIE-16GB
Свободная память: 0.59 GB
Используется устройство: cuda:0
GPU: Tesla P100-PCIE-16GB
Всего памяти: 17.06 GB
Текущее использование: Alloc: 1.20GB, Reserved: 16.48GB
Сбор обучающих изображений...
Всего найдено 9889 изображений

Fold 1/5
Train samples: 7911, Val samples: 1978
Создание модели...


Fold 1 Epoch 1/15 [Val]: 100%|██████████| 248/248 [00:23<00:00, 10.39it/s, loss=0.1137, acc=91.81%]


Fold 1 Epoch 1: Train Acc: 81.30%, Val Acc: 91.81%


Fold 1 Epoch 2/15 [Val]: 100%|██████████| 248/248 [00:22<00:00, 11.26it/s, loss=0.1327, acc=89.64%]


Fold 1 Epoch 2: Train Acc: 92.93%, Val Acc: 89.64%


...

Fold 4 Epoch 10/15 [Val]: 100%|██████████| 248/248 [00:22<00:00, 11.23it/s, loss=0.0895, acc=95.30%]


Fold 4 Epoch 10: Train Acc: 99.84%, Val Acc: 95.30%


Fold 4 Epoch 11/15 [Val]: 100%|██████████| 248/248 [00:22<00:00, 11.26it/s, loss=0.0720, acc=96.01%]


Fold 4 Epoch 11: Train Acc: 99.87%, Val Acc: 96.01%


Fold 4 Epoch 12/15 [Train]:   5%|▍         | 45/989 [00:34<11:40,  1.35it/s, loss=0.0000, acc=100.00%, mem=Alloc: 3.66GB, Reserved: 7.66GB]

In [2]:
import torch
import timm
import pandas as pd
from pathlib import Path
from PIL import Image
import torchvision.transforms as transforms
from tqdm import tqdm
import os
import numpy as np

# Те же константы и классы
CLASS_NAMES = [
    "Апельсин", "Бананы", "Груши", "Кабачки", "Капуста", 
    "Картофель", "Киви", "Лимон", "Лук", "Мандарины", 
    "Морковь", "Огурцы", "Томаты", "Яблоки зелёные", "Яблоки красные"
]
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = 256
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Используется устройство: {DEVICE}")

# Трансформации
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Загружаем 3 сохраненные модели с правильной архитектурой
models = []
for fold in range(1, 6):
    print(f"Загрузка модели fold {fold}...")
    
    # Используем ConvNeXt V2, как в оригинальном коде
    try:
        model = timm.create_model(
            'convnextv2_base.fcmae_ft_in22k_in1k',  # Важно: используем V2 версию!
            pretrained=False, 
            num_classes=NUM_CLASSES
        )
    except:
        # Если V2 не доступен, используем обычный ConvNeXt с параметром strict=False
        print("ConvNeXt V2 не доступен, пробуем обычный ConvNeXt...")
        model = timm.create_model(
            'convnext_base.fb_in22k_ft_in1k',
            pretrained=False, 
            num_classes=NUM_CLASSES
        )
    
    # Загружаем веса с игнорированием несоответствий
    state_dict = torch.load(f'best_model_fold_{fold}.pth', map_location='cpu')
    
    # Пробуем загрузить с strict=False
    try:
        model.load_state_dict(state_dict, strict=False)
        print(f"  Модель {fold} загружена с параметром strict=False")
    except Exception as e:
        print(f"  Ошибка при загрузке: {e}")
        continue
    
    model = model.to(DEVICE)
    model.eval()
    models.append(model)
    print(f"  Модель {fold} готова к работе")

print(f"\nЗагружено {len(models)} моделей")

if len(models) == 0:
    print("НЕ УДАЛОСЬ ЗАГРУЗИТЬ НИ ОДНОЙ МОДЕЛИ!")
    exit()

# Предсказание
test_path = "/kaggle/input/dl-lab-1-image-classification/test_images/test_images"
test_images = list(Path(test_path).glob('*.*'))
print(f"Найдено {len(test_images)} тестовых изображений")

predictions = []

for img_path in tqdm(test_images, desc="Предсказание"):
    try:
        image = Image.open(img_path).convert('RGB')
        image_tensor = test_transform(image).unsqueeze(0).to(DEVICE)
        
        with torch.no_grad():
            # Собираем предсказания от всех моделей
            all_outputs = []
            for model in models:
                outputs = model(image_tensor)
                all_outputs.append(outputs)
            
            # Усредняем
            avg_output = torch.mean(torch.stack(all_outputs), dim=0)
            pred = torch.argmax(avg_output, dim=1).item()
        
        predictions.append({'image_id': img_path.name, 'label': pred})
        
    except Exception as e:
        print(f"Ошибка при обработке {img_path}: {e}")
        predictions.append({'image_id': img_path.name, 'label': 0})

# Сохраняем результат
df = pd.DataFrame(predictions)
df.to_csv('test_predictions.csv', index=False)
print("\nПредсказания сохранены в test_predictions.csv")

# Показываем статистику
print("\nРаспределение предсказаний:")
for i, class_name in enumerate(CLASS_NAMES):
    count = len(df[df['label'] == i])
    if count > 0:
        print(f"  {class_name}: {count}")

# Показываем первые 10 предсказаний
print("\nПервые 10 предсказаний:")
print(df.head(10))

Используется устройство: cuda
Загрузка модели fold 1...
  Модель 1 загружена с параметром strict=False
  Модель 1 готова к работе
Загрузка модели fold 2...
  Модель 2 загружена с параметром strict=False
  Модель 2 готова к работе
Загрузка модели fold 3...
  Модель 3 загружена с параметром strict=False
  Модель 3 готова к работе

Загружено 3 моделей
Найдено 2503 тестовых изображений


Предсказание: 100%|██████████| 2503/2503 [02:33<00:00, 16.34it/s]


Предсказания сохранены в test_predictions.csv

Распределение предсказаний:
  Апельсин: 225
  Бананы: 207
  Груши: 105
  Кабачки: 74
  Капуста: 206
  Картофель: 203
  Киви: 47
  Лимон: 167
  Лук: 168
  Мандарины: 189
  Морковь: 169
  Огурцы: 156
  Томаты: 197
  Яблоки зелёные: 214
  Яблоки красные: 176

Первые 10 предсказаний:
                               image_id  label
0  21cebf8372a04f2d99fefb5fa91be529.jpg      0
1  a7c13e7c90784c499a87f5ff24b47e6d.jpg      2
2  3034b2f6538c48bca4de5b1fc888769a.jpg      5
3  db70a81660804ae880ac9a2ca4ebd4ab.jpg      0
4  1e3cacd8a38c4cb79afa113c6a5550f2.jpg      0
5  bc46a9a69b5d4bafac2b5ad5640373ef.jpg     13
6  f837c85941a948cab590faafa10f07e0.jpg      1
7  8f707f3b642f474f9255f934abe71ef8.jpg      5
8  8157f57b6aee4a988519a798e043c7a4.jpg     11
9  60be5b872bfe414290658f4fd928c5c8.jpg      5


In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
from pathlib import Path
import timm
import gc
import warnings
warnings.filterwarnings('ignore')

# Константы
CLASS_NAMES = [
    "Апельсин", "Бананы", "Груши", "Кабачки", "Капуста", 
    "Картофель", "Киви", "Лимон", "Лук", "Мандарины", 
    "Морковь", "Огурцы", "Томаты", "Яблоки зелёные", "Яблоки красные"
]
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS_FINETUNE = 5  # Меньше эпох для тонкой настройки
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class_to_idx = {cls_name: idx for idx, cls_name in enumerate(CLASS_NAMES)}

print(f"Используется устройство: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")
    torch.cuda.empty_cache()
    gc.collect()

class CustomDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        try:
            image = Image.open(image_path).convert('RGB')
        except:
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color='black')
        
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

def get_train_transforms(is_train=True):
    if is_train:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

def collect_train_images(train_path):
    image_paths = []
    labels = []
    train_path = Path(train_path)
    
    for class_name in CLASS_NAMES:
        class_folder = train_path / class_name
        if not class_folder.exists():
            continue
        
        for subfolder in class_folder.iterdir():
            if subfolder.is_dir():
                for img_file in subfolder.glob('*.*'):
                    if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                        image_paths.append(str(img_file))
                        labels.append(class_to_idx[class_name])
    
    return image_paths, labels

def finetune_model(model, train_loader, val_loader, fold):
    """Тонкая настройка модели"""
    # Замораживаем все слои кроме последних
    for name, param in model.named_parameters():
        if 'head' not in name and 'stages.3' not in name:  # Размораживаем только head и последнюю стадию
            param.requires_grad = False
    
    # Считаем количество trainable параметров
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable_params}/{total_params} ({100*trainable_params/total_params:.2f}%)")
    
    # Оптимизатор только для trainable параметров
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), 
        lr=5e-5,  # Маленький learning rate для тонкой настройки
        weight_decay=0.01
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FINETUNE)
    criterion = nn.CrossEntropyLoss()
    
    best_val_acc = 0.0
    
    for epoch in range(EPOCHS_FINETUNE):
        # Training
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        train_bar = tqdm(train_loader, desc=f'Fold {fold} Finetune Epoch {epoch+1}/{EPOCHS_FINETUNE} [Train]')
        for images, labels in train_bar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            train_bar.set_postfix({
                'loss': f'{train_loss/train_total:.4f}',
                'acc': f'{100.*train_correct/train_total:.2f}%'
            })
        
        train_acc = 100. * train_correct / train_total
        
        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f'Fold {fold} Finetune Epoch {epoch+1}/{EPOCHS_FINETUNE} [Val]')
            for images, labels in val_bar:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
                val_bar.set_postfix({'acc': f'{100.*val_correct/val_total:.2f}%'})
        
        val_acc = 100. * val_correct / val_total
        scheduler.step()
        
        print(f'Fold {fold} Epoch {epoch+1}: Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%')
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f'finetuned_model_fold_{fold}.pth')
            print(f"  -> Saved best model with val_acc: {val_acc:.2f}%")
    
    return best_val_acc

def main():
    # Пути к данным
    train_path = "/kaggle/input/dl-lab-1-image-classification/train/train"
    
    # Собираем все изображения
    print("Сбор обучающих изображений...")
    image_paths, labels = collect_train_images(train_path)
    print(f"Всего найдено {len(image_paths)} изображений")
    
    # Загружаем три сохраненные модели
    models = []
    folds_to_finetune = [1, 2, 3]  # Только первые три фолда
    
    for fold in folds_to_finetune:
        print(f"\n{'='*50}")
        print(f"Загрузка и тонкая настройка модели Fold {fold}")
        print(f"{'='*50}")
        
        # Создаем модель (ConvNeXt V2)
        try:
            model = timm.create_model(
                'convnextv2_base.fcmae_ft_in22k_in1k',
                pretrained=False,
                num_classes=NUM_CLASSES
            )
        except:
            print("ConvNeXt V2 не доступен, используем обычный ConvNeXt")
            model = timm.create_model(
                'convnext_base.fb_in22k_ft_in1k',
                pretrained=False,
                num_classes=NUM_CLASSES
            )
        
        # Загружаем сохраненные веса
        model_path = f'best_model_fold_{fold}.pth'
        if os.path.exists(model_path):
            state_dict = torch.load(model_path, map_location='cpu')
            model.load_state_dict(state_dict, strict=False)
            print(f"Загружены веса из {model_path}")
        else:
            print(f"Файл {model_path} не найден!")
            continue
        
        model = model.to(DEVICE)
        
        # Создаем DataLoader для этого фолда (используем все данные для тонкой настройки)
        # Но для валидации возьмем часть данных
        from sklearn.model_selection import train_test_split
        
        # Разделяем данные для этого фолда
        train_paths_fold, val_paths_fold, train_labels_fold, val_labels_fold = train_test_split(
            image_paths, labels, test_size=0.2, random_state=42+fold, stratify=labels
        )
        
        train_dataset = CustomDataset(
            train_paths_fold, 
            train_labels_fold, 
            transform=get_train_transforms(is_train=True)
        )
        val_dataset = CustomDataset(
            val_paths_fold, 
            val_labels_fold, 
            transform=get_train_transforms(is_train=False)
        )
        
        train_loader = DataLoader(
            train_dataset, 
            batch_size=BATCH_SIZE, 
            shuffle=True, 
            num_workers=2,
            pin_memory=True
        )
        val_loader = DataLoader(
            val_dataset, 
            batch_size=BATCH_SIZE, 
            shuffle=False, 
            num_workers=2,
            pin_memory=True
        )
        
        # Тонкая настройка
        best_val_acc = finetune_model(model, train_loader, val_loader, fold)
        models.append(model)
        
        # Очистка памяти
        torch.cuda.empty_cache()
        gc.collect()
    
    print(f"\nТонкая настройка завершена. Дообучено {len(models)} моделей.")
    
    # Предсказание на тестовых данных
    print("\n" + "="*50)
    print("Предсказание на тестовых изображениях")
    print("="*50)
    
    test_path = "/kaggle/input/dl-lab-1-image-classification/test_images/test_images"
    test_images = list(Path(test_path).glob('*.*'))
    print(f"Найдено {len(test_images)} тестовых изображений")
    
    test_transform = get_train_transforms(is_train=False)
    predictions = []
    
    for img_path in tqdm(test_images, desc="Предсказание"):
        try:
            image = Image.open(img_path).convert('RGB')
            image_tensor = test_transform(image).unsqueeze(0).to(DEVICE)
            
            with torch.no_grad():
                # Собираем предсказания от всех моделей
                all_outputs = []
                for model in models:
                    model.eval()
                    outputs = model(image_tensor)
                    all_outputs.append(outputs)
                
                # Усредняем
                avg_output = torch.mean(torch.stack(all_outputs), dim=0)
                probs = torch.softmax(avg_output, dim=1)
                pred = torch.argmax(probs, dim=1).item()
                confidence = probs[0][pred].item()
            
            predictions.append({
                'image_id': img_path.name, 
                'label': pred,
                'confidence': confidence
            })
            
        except Exception as e:
            print(f"Ошибка при обработке {img_path}: {e}")
            predictions.append({'image_id': img_path.name, 'label': 0, 'confidence': 0.0})
    
    # Сохраняем результаты
    df = pd.DataFrame(predictions)
    df[['image_id', 'label']].to_csv('finetuned_predictions.csv', index=False)
    
    # Сохраняем расширенный результат с confidence
    df.to_csv('finetuned_predictions_detailed.csv', index=False)
    
    print("\n" + "="*50)
    print("РЕЗУЛЬТАТЫ")
    print("="*50)
    print(f"Предсказания сохранены в finetuned_predictions.csv")
    
    # Статистика
    print("\nРаспределение предсказаний:")
    label_counts = df['label'].value_counts().sort_index()
    for label, count in label_counts.items():
        if count > 0:
            print(f"  {CLASS_NAMES[label]}: {count}")
    
    # Средняя уверенность
    avg_confidence = df['confidence'].mean()
    print(f"\nСредняя уверенность модели: {avg_confidence:.2%}")
    
    # Показываем первые 10 предсказаний
    print("\nПервые 10 предсказаний:")
    for idx, row in df.head(10).iterrows():
        print(f"  {row['image_id']}: {CLASS_NAMES[row['label']]} (conf: {row['confidence']:.2%})")

if __name__ == "__main__":
    main()

Используется устройство: cuda
GPU: Tesla P100-PCIE-16GB
Сбор обучающих изображений...
Всего найдено 9889 изображений

Загрузка и тонкая настройка модели Fold 1
Загружены веса из best_model_fold_1.pth
Trainable parameters: 27482127/87708175 (31.33%)


Fold 1 Finetune Epoch 1/5 [Val]: 100%|██████████| 248/248 [00:23<00:00, 10.49it/s, acc=98.84%]


Fold 1 Epoch 1: Train Acc: 98.84%, Val Acc: 98.84%
  -> Saved best model with val_acc: 98.84%


Fold 1 Finetune Epoch 2/5 [Val]: 100%|██████████| 248/248 [00:21<00:00, 11.37it/s, acc=98.79%]


Fold 1 Epoch 2: Train Acc: 99.44%, Val Acc: 98.79%


...

Fold 3 Epoch 4: Train Acc: 99.81%, Val Acc: 99.29%


Fold 3 Finetune Epoch 5/5 [Val]: 100%|██████████| 248/248 [00:21<00:00, 11.36it/s, acc=99.24%]


Fold 3 Epoch 5: Train Acc: 99.87%, Val Acc: 99.24%

Тонкая настройка завершена. Дообучено 3 моделей.

Предсказание на тестовых изображениях
Найдено 2503 тестовых изображений


Предсказание: 100%|██████████| 2503/2503 [02:19<00:00, 17.93it/s]



РЕЗУЛЬТАТЫ
Предсказания сохранены в finetuned_predictions.csv

Распределение предсказаний:
  Апельсин: 225
  Бананы: 207
  Груши: 107
  Кабачки: 74
  Капуста: 206
  Картофель: 198
  Киви: 49
  Лимон: 167
  Лук: 170
  Мандарины: 188
  Морковь: 170
  Огурцы: 156
  Томаты: 198
  Яблоки зелёные: 213
  Яблоки красные: 175

Средняя уверенность модели: 98.62%

Первые 10 предсказаний:
  21cebf8372a04f2d99fefb5fa91be529.jpg: Апельсин (conf: 99.94%)
  a7c13e7c90784c499a87f5ff24b47e6d.jpg: Груши (conf: 99.93%)
  3034b2f6538c48bca4de5b1fc888769a.jpg: Картофель (conf: 99.98%)
  db70a81660804ae880ac9a2ca4ebd4ab.jpg: Апельсин (conf: 100.00%)
  1e3cacd8a38c4cb79afa113c6a5550f2.jpg: Апельсин (conf: 99.99%)
  bc46a9a69b5d4bafac2b5ad5640373ef.jpg: Яблоки зелёные (conf: 99.99%)
  f837c85941a948cab590faafa10f07e0.jpg: Бананы (conf: 99.99%)
  8f707f3b642f474f9255f934abe71ef8.jpg: Картофель (conf: 99.99%)
  8157f57b6aee4a988519a798e043c7a4.jpg: Огурцы (conf: 100.00%)
  60be5b872bfe414290658f4fd928c5c8.jpg: 

In [7]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
from pathlib import Path
import timm
import gc
import warnings
warnings.filterwarnings('ignore')

# Константы
CLASS_NAMES = [
    "Апельсин", "Бананы", "Груши", "Кабачки", "Капуста", 
    "Картофель", "Киви", "Лимон", "Лук", "Мандарины", 
    "Морковь", "Огурцы", "Томаты", "Яблоки зелёные", "Яблоки красные"
]
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = 256
BATCH_SIZE = 8
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Используется устройство: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")
    torch.cuda.empty_cache()
    gc.collect()

class CustomDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        try:
            image = Image.open(image_path).convert('RGB')
        except Exception as e:
            print(f"Ошибка загрузки {image_path}: {e}")
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color='black')
        
        if self.transform:
            image = self.transform(image)
        
        return image, str(image_path)

# Базовые трансформации для нормализации
base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# TTA трансформации (все включают ToTensor и Normalize)
def get_tta_transforms():
    """Возвращает список трансформаций для TTA"""
    tta_transforms = []
    
    # 1. Оригинал
    tta_transforms.append(transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]))
    
    # 2. Горизонтальное отражение
    tta_transforms.append(transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]))
    
    # 3. Вертикальное отражение
    tta_transforms.append(transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomVerticalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]))
    
    # 4. Поворот на 90 градусов
    tta_transforms.append(transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(90),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]))
    
    # 5. Поворот на 180 градусов
    tta_transforms.append(transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(180),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]))
    
    # 6. Поворот на 270 градусов
    tta_transforms.append(transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(270),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]))
    
    return tta_transforms

def predict_with_tta(model, image_tensor, tta_transforms_list):
    """Предсказание с TTA для тензора изображения"""
    model.eval()
    all_probs = []
    
    with torch.no_grad():
        for tta_transform in tta_transforms_list:
            # Для TTA нам нужно применять трансформации к PIL изображению
            # Но у нас уже тензор, поэтому этот подход не работает
            # Изменим логику: будем передавать PIL изображение
            pass
    
    return None

def main():
    print("\n" + "="*60)
    print("ЗАГРУЗКА МОДЕЛЕЙ И ПРЕДСКАЗАНИЕ С TTA (TEST TIME AUGMENTATION)")
    print("="*60)
    
    # Загружаем дообученные модели
    models = []
    folds_to_load = [1, 2, 3]  # Загружаем три модели
    
    for fold in folds_to_load:
        print(f"\nЗагрузка модели Fold {fold}...")
        
        # Создаем модель (такой же архитектуры)
        try:
            model = timm.create_model(
                'convnextv2_base.fcmae_ft_in22k_in1k',
                pretrained=False,
                num_classes=NUM_CLASSES
            )
        except:
            print("ConvNeXt V2 не доступен, используем обычный ConvNeXt")
            model = timm.create_model(
                'convnext_base.fb_in22k_ft_in1k',
                pretrained=False,
                num_classes=NUM_CLASSES
            )
        
        # Загружаем дообученные веса
        model_path = f'finetuned_model_fold_{fold}.pth'
        if os.path.exists(model_path):
            state_dict = torch.load(model_path, map_location='cpu')
            model.load_state_dict(state_dict, strict=True)
            print(f"✓ Загружены дообученные веса из {model_path}")
        else:
            # Если нет дообученных, пробуем обычные
            model_path = f'best_model_fold_{fold}.pth'
            if os.path.exists(model_path):
                state_dict = torch.load(model_path, map_location='cpu')
                model.load_state_dict(state_dict, strict=False)
                print(f"✓ Загружены веса из {model_path}")
            else:
                print(f"✗ Файл {model_path} не найден!")
                continue
        
        model = model.to(DEVICE)
        model.eval()
        models.append(model)
        
        print(f"Модель Fold {fold} загружена и переведена в режим eval")
    
    if not models:
        print("Не удалось загрузить ни одной модели!")
        return
    
    print(f"\n✓ Загружено {len(models)} моделей для ансамбля с TTA")
    
    # Получаем TTA трансформации
    tta_transforms_list = get_tta_transforms()
    print(f"✓ Используется {len(tta_transforms_list)} аугментаций для TTA")
    
    # Тестовые изображения
    test_path = "/kaggle/input/dl-lab-1-image-classification/test_images/test_images"
    test_images = sorted(Path(test_path).glob('*.*'))
    print(f"\nНайдено {len(test_images)} тестовых изображений")
    
    # Предсказания
    predictions = []
    
    print("\n" + "="*60)
    print("ВЫПОЛНЕНИЕ ПРЕДСКАЗАНИЙ С TTA")
    print("="*60)
    
    for img_path in tqdm(test_images, desc="TTA Prediction"):
        img_name = img_path.name
        
        try:
            # Загружаем изображение
            pil_image = Image.open(img_path).convert('RGB')
            
            # Собираем предсказания от всех моделей с TTA
            all_model_probs = []
            
            for model in models:
                model.eval()
                model_probs = []
                
                with torch.no_grad():
                    # Применяем все TTA трансформации
                    for tta_transform in tta_transforms_list:
                        # Применяем трансформацию к PIL изображению
                        augmented_tensor = tta_transform(pil_image).unsqueeze(0).to(DEVICE)
                        
                        # Прямой проход
                        outputs = model(augmented_tensor)
                        probs = torch.softmax(outputs, dim=1)
                        model_probs.append(probs.cpu())
                
                # Усредняем по аугментациям для текущей модели
                avg_model_probs = torch.mean(torch.stack(model_probs), dim=0)
                all_model_probs.append(avg_model_probs)
            
            # Усредняем по моделям
            avg_probs = torch.mean(torch.stack(all_model_probs), dim=0)
            
            # Получаем предсказание и уверенность
            pred = torch.argmax(avg_probs, dim=1).item()
            confidence = avg_probs[0][pred].item()
            
            # Сохраняем все вероятности для анализа
            probs_dict = {CLASS_NAMES[i]: float(avg_probs[0][i]) for i in range(NUM_CLASSES)}
            top3 = sorted(probs_dict.items(), key=lambda x: x[1], reverse=True)[:3]
            
            predictions.append({
                'image_id': img_name,
                'label': pred,
                'class_name': CLASS_NAMES[pred],
                'confidence': confidence,
                'top1': f"{top3[0][0]}: {top3[0][1]:.2%}",
                'top2': f"{top3[1][0]}: {top3[1][1]:.2%}",
                'top3': f"{top3[2][0]}: {top3[2][1]:.2%}"
            })
            
        except Exception as e:
            print(f"Ошибка при обработке {img_name}: {e}")
            predictions.append({
                'image_id': img_name,
                'label': 0,
                'class_name': CLASS_NAMES[0],
                'confidence': 0.0,
                'top1': 'Error',
                'top2': 'Error',
                'top3': 'Error'
            })
    
    # Сохраняем результаты
    df = pd.DataFrame(predictions)
    
    # Простой формат для сабмишна
    df[['image_id', 'label']].to_csv('tta_predictions.csv', index=False)
    
    # Детальный формат с TTA информацией
    df.to_csv('tta_predictions_detailed.csv', index=False)
    
    print("\n" + "="*60)
    print("РЕЗУЛЬТАТЫ TTA")
    print("="*60)
    print(f"✓ Предсказания сохранены в:")
    print(f"  - tta_predictions.csv (простой формат)")
    print(f"  - tta_predictions_detailed.csv (детальный формат с TTA)")
    
    # Статистика
    print("\n📊 Распределение предсказаний:")
    label_counts = df['label'].value_counts().sort_index()
    for label, count in label_counts.items():
        if count > 0:
            print(f"  {CLASS_NAMES[label]}: {count} ({count/len(df)*100:.1f}%)")
    
    # Средняя уверенность
    avg_confidence = df['confidence'].mean()
    print(f"\n📈 Средняя уверенность модели с TTA: {avg_confidence:.2%}")
    
    # Сравнение с обычным предсказанием (если есть файл)
    if os.path.exists('finetuned_predictions.csv'):
        df_regular = pd.read_csv('finetuned_predictions.csv')
        common_ids = set(df['image_id']) & set(df_regular['image_id'])
        
        if common_ids:
            df_regular_subset = df_regular[df_regular['image_id'].isin(common_ids)]
            df_tta_subset = df[df['image_id'].isin(common_ids)]
            
            # Сравниваем предсказания
            comparison = pd.merge(
                df_regular_subset[['image_id', 'label']], 
                df_tta_subset[['image_id', 'label', 'confidence']], 
                on='image_id', 
                suffixes=('_regular', '_tta')
            )
            
            # Считаем совпадения
            matches = (comparison['label_regular'] == comparison['label_tta']).sum()
            print(f"\n🔄 Сравнение с обычным предсказанием:")
            print(f"  Совпадение предсказаний: {matches}/{len(comparison)} ({matches/len(comparison)*100:.1f}%)")
            
            # Где TTA изменил предсказание
            changed = comparison[comparison['label_regular'] != comparison['label_tta']]
            if len(changed) > 0:
                print(f"\n  Измененные предсказания (TTA vs Regular):")
                for _, row in changed.head(5).iterrows():
                    print(f"    {row['image_id']}: {CLASS_NAMES[row['label_regular']]} → {CLASS_NAMES[row['label_tta']]} (conf: {row['confidence_tta']:.2%})")
    
    # Показываем первые 10 предсказаний
    print("\n🔍 Первые 10 предсказаний с TTA:")
    for idx, row in df.head(10).iterrows():
        print(f"  {row['image_id']}: {row['class_name']}")
        print(f"    уверенность: {row['confidence']:.2%}")
        print(f"    Top-3: {row['top1']}, {row['top2']}, {row['top3']}")
        print()

if __name__ == "__main__":
    main()

Используется устройство: cuda
GPU: Tesla P100-PCIE-16GB

ЗАГРУЗКА МОДЕЛЕЙ И ПРЕДСКАЗАНИЕ С TTA (TEST TIME AUGMENTATION)

Загрузка модели Fold 1...
✓ Загружены дообученные веса из finetuned_model_fold_1.pth
Модель Fold 1 загружена и переведена в режим eval

Загрузка модели Fold 2...
✓ Загружены дообученные веса из finetuned_model_fold_2.pth
Модель Fold 2 загружена и переведена в режим eval

Загрузка модели Fold 3...
✓ Загружены дообученные веса из finetuned_model_fold_3.pth
Модель Fold 3 загружена и переведена в режим eval

✓ Загружено 3 моделей для ансамбля с TTA
✓ Используется 6 аугментаций для TTA

Найдено 2503 тестовых изображений

ВЫПОЛНЕНИЕ ПРЕДСКАЗАНИЙ С TTA


TTA Prediction: 100%|██████████| 2503/2503 [14:16<00:00,  2.92it/s]


РЕЗУЛЬТАТЫ TTA
✓ Предсказания сохранены в:
  - tta_predictions.csv (простой формат)
  - tta_predictions_detailed.csv (детальный формат с TTA)

📊 Распределение предсказаний:
  Апельсин: 226 (9.0%)
  Бананы: 209 (8.3%)
  Груши: 102 (4.1%)
  Кабачки: 71 (2.8%)
  Капуста: 206 (8.2%)
  Картофель: 202 (8.1%)
  Киви: 45 (1.8%)
  Лимон: 168 (6.7%)
  Лук: 171 (6.8%)
  Мандарины: 188 (7.5%)
  Морковь: 171 (6.8%)
  Огурцы: 157 (6.3%)
  Томаты: 197 (7.9%)
  Яблоки зелёные: 218 (8.7%)
  Яблоки красные: 172 (6.9%)

📈 Средняя уверенность модели с TTA: 95.40%

🔄 Сравнение с обычным предсказанием:
  Совпадение предсказаний: 2472/2503 (98.8%)

  Измененные предсказания (TTA vs Regular):


KeyError: 'confidence_tta'